# Masking delle feature FC in funzione della lesione

## Il problema, in parole semplici

Ogni paziente stroke ha due tipi di dati sul cervello:

- **La lesione**: la zona di tessuto danneggiato dall'ictus (un'immagine 3D che dice, voxel per voxel, "qui c'e' danno / qui no").
- **La connettivita' funzionale (FC)**: quanto "comunicano tra loro" le diverse zone del cervello a riposo, misurata con la risonanza funzionale. Il cervello viene diviso in centinaia di piccole zone ("nodi" o "parcel"), e per ogni coppia di zone si misura quanto la loro attivita' e' sincronizzata nel tempo. Il risultato e' una grande tabella nodo x nodo (una matrice), gia' calcolata e disponibile per i nostri pazienti WashU.

**Il problema**: se una zona e' dentro la lesione (tessuto morto/danneggiato), il segnale che ne esce non e' piu' "attivita' cerebrale reale" - e' rumore o assenza di segnale. Se non lo togliamo, rischiamo di scambiare "questa zona e' distrutta" per "comunicazione alterata tra zone sane". Serve quindi **marcare come mancante** ogni valore di connettivita' che coinvolge una zona troppo danneggiata, prima di poter usare questi dati per collegare lesione/disconnessione ai deficit clinici (linguaggio, attenzione, motricita'...).

Questo notebook fa esattamente questo, su piu' pazienti, e verifica il risultato su dati reali (non simulati) prima di trasformarlo in codice definitivo.

## Da dove viene il metodo

Non e' un metodo inventato per l'occasione - e' lo stesso approccio gia' usato in letteratura su questa stessa coorte di pazienti (WashU, laboratorio Corbetta):

- **Siegel et al. 2016, *PNAS*** — le connessioni delle zone dentro la lesione vengono rimosse dalle analisi (o azzerate, quando lo strumento a valle non accetta valori mancanti).
- **Griffis et al. 2019, *Neuron*** (gia' salvato in `assets/papers/`) — versione piu' precisa: una zona viene esclusa solo se **una percentuale sufficiente** del suo territorio e' dentro la lesione. Soglia standard di campo: **50%**. Punto chiave (citazione esatta dal paper): *"Because the PLSC approach cannot accommodate missing values, functional connectivity between parcels that had been excluded ... was set to 0"* — cioe' il valore mancante viene marcato come tale (NaN) e sostituito con un valore concreto **solo nel momento in cui serve** a uno strumento che non tollera dati mancanti, non prima.
- **XCP-D** (`xcp_d/interfaces/connectivity.py`, classe `NiftiParcellate`) — il software che ha generato le nostre matrici FC ha gia' un meccanismo identico (`min_coverage`, soglia di default 0.5), per un problema diverso (copertura BOLD). Riusiamo lo stesso meccanismo con la lesione al posto del problema originale.

**Decisioni prese con l'utente**:
1. Niente funzione di conteggio scritta a mano — si riusa `nilearn.maskers.NiftiLabelsMasker`, stesso schema di XCP-D.
2. Le zone compromesse vengono marcate con **NaN**, non azzerate direttamente — l'azzeramento (o un'altra strategia) avviene solo più avanti, come passo separato ed esplicito, subito prima di un metodo (PCA/UMAP) che non tollera NaN.
3. Le matrici mascherate (con NaN) vengono **salvate su disco così come sono**, prima di essere vettorizzate — sono un formato ispezionabile a occhio, utile per il controllo di qualità, indipendente da come verranno poi impilate/vettorizzate.

**Destinazioni concordate su disco**:
- `data/derived/features/masked_fc/` — matrici mascherate per singolo paziente (formato tabellare, con NaN).
- `data/derived/features/fc_matrix/` — la matrice 2D finale (pazienti x connessioni), impilata e vettorizzata.

In [1]:
import warnings

# nilearn/nibabel emettono warning di deprecazione non rilevanti per questa analisi - silenziati
# solo per leggibilita' dell'output, non nascondono errori reali (quelli restano ValueError/AssertionError).
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import glob  # per trovare il file CSV della matrice FC senza scrivere il nome esatto a mano
from pathlib import Path

import nibabel as nib  # legge/scrive immagini cerebrali in formato NIfTI (.nii.gz)
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img  # riallinea due immagini 3D sulla stessa griglia spaziale
from nilearn.maskers import NiftiLabelsMasker  # estrae valori per-zona da un'immagine, data una mappa di zone

# --- Parametri di questa run: piu' pazienti (dimostrativi), una versione della mappa cerebrale ---
SUBJECTS = ["sub-STUNIPD0002", "sub-STUNIPD0003", "sub-STUNIPD0006"]  # lesioni diverse, per mostrare pattern diversi di masking
ATLAS_COMBO = "atlas-Yan200TianS2Buckner7N"  # una delle 12 mappe cerebrali disponibili (200 zone corticali + subcortex + cervelletto)
MIN_COVERAGE = 0.5  # soglia: sotto il 50% di territorio sano, la zona e' "compromessa" (standard di campo, vedi sopra)

DATA_ROOT = "../data/clinical_connectome/derivatives/UNIPD/WashU"  # dati del paziente (lesione + connettivita')
ATLAS_ROOT = f"../assets/atlases/fmriprep/{ATLAS_COMBO}"  # mappa cerebrale di riferimento (copiata dal server)

# destinazioni finali concordate - create qui se non esistono ancora, mai assunte gia' presenti
MASKED_FC_ROOT = Path(f"../data/derived/features/masked_fc/{ATLAS_COMBO}")
FC_MATRIX_ROOT = Path(f"../data/derived/features/fc_matrix/{ATLAS_COMBO}")
MASKED_FC_ROOT.mkdir(parents=True, exist_ok=True)
FC_MATRIX_ROOT.mkdir(parents=True, exist_ok=True)

## 1. La mappa del cervello ("atlante")

Per sapere "quanto e' dentro la lesione" ogni singola zona, serve prima una **mappa di riferimento** che dice esattamente dove sono i confini di ciascuna delle ~240 zone usate per calcolare la connettivita'. Senza questa mappa non sappiamo a quale zona appartiene ogni punto del cervello. Questo passo va fatto **una sola volta** (non per ogni paziente): la mappa e' sempre la stessa, cambia solo la lesione da confrontarci.

**Cosa contiene questa mappa** (gia' pronta sul server EBRAIN, cartella `Atlases/fmriprep/atlas-Yan200TianS2Buckner7N/`, copiata in locale):
- Un'immagine 3D (`*_res-2_dseg.nii.gz`) dove ogni punto del cervello ha un numero: quel numero identifica a quale delle ~240 zone appartiene quel punto (0 = fuori dal cervello).
- Una tabella (`*_dseg.tsv`) che traduce ogni numero in un nome leggibile (es. "corteccia visiva sinistra").

Usiamo direttamente la versione a **2mm** di risoluzione (`res-2`) perche' e' gia' alla stessa risoluzione della lesion mask WashU — evita un passaggio di ricampionamento in piu'.

In [2]:
# squeeze_image: il file ha una dimensione extra inutile (4 dimensioni invece di 3), la togliamo
atlas_img = nib.squeeze_image(
    nib.load(f"{ATLAS_ROOT}/{ATLAS_COMBO}_space-MNI152NLin6Asym_res-2_dseg.nii.gz")
)

# la tabella che traduce ogni numero-zona nel suo nome (es. 1 -> "7Networks_LH_Default_IPL_1")
label_table = pd.read_csv(f"{ATLAS_ROOT}/{ATLAS_COMBO}_dseg.tsv", sep="\t")
label_ids = label_table["index"].tolist()  # tutti i numeri-zona attesi, nell'ordine ufficiale
id_to_name = dict(zip(label_table["index"], label_table["label"]))  # dizionario numero -> nome
node_names = np.array([id_to_name[i] for i in label_ids])  # nomi leggibili delle zone, nell'ordine ufficiale

print(f"Atlante caricato: {len(label_ids)} nodi")
label_table.head(3)  # anteprima: come si presenta la tabella

Atlante caricato: 239 nodi


,index,label
0,1,7Networks_LH_Default_IPL_1
1,2,7Networks_LH_Default_IPL_2
2,3,7Networks_LH_Default_IPL_3


## Fase A — un paziente alla volta

Per ciascun paziente, ripetiamo sempre la stessa sequenza. Esempio con un atlante giocattolo a 4 zone (A, B, C, D) e un paziente con lesione all'80% su C e al 30% su D, per rendere concreti i passaggi prima di vederli sui dati veri (239 zone):

1. **Carica la lesione** del paziente.
2. **Riallinea la lesione alla griglia dell'atlante** (`resample_to_img`, nearest neighbor) — lesione e atlante sono due file creati separatamente: anche a parita' di dimensioni, l'orientamento spaziale reale va sempre verificato e corretto, mai assunto identico (vedi affine invertito piu' sotto).
3. **Calcola la percentuale di zona sana**, per ciascuna delle 4 zone: A=100%, B=100%, C=20% (compromessa), D=70% (ancora valida).
4. **Decide quali zone sono compromesse** (soglia 50%): solo C.
5. **Carica la matrice di connettivita'** del paziente (una tabella 4x4: A-B, A-C, A-D, B-C, B-D, C-D) e verifica che i nomi delle zone combacino con l'atlante.
6. **Marca con NaN** tutta la riga e tutta la colonna di C (compromessa) — A, B, D restano intatte.
7. **Salva su disco** questa matrice 4x4 mascherata, cosi' com'e' — un formato leggibile, utile per controllo qualita', indipendente da come verra' poi usata.
8. **Vettorizza**: tiene solo le 6 connessioni uniche (A-B, A-C, A-D, B-C, B-D, C-D — niente doppioni come B-A, niente diagonale) in un unico vettore, di cui 3 ora sono NaN (quelle che coinvolgono C).

In [3]:
def compute_parcel_coverage(atlas_img, label_ids, healthy_img):
    """Frazione di voxel sani per ciascuna zona, nello stesso ordine di label_ids.

    Una zona con zero voxel sani rimasti viene rimossa da NiftiLabelsMasker quando gli si
    passa mask_img (non restituita come 0) - gestito qui esplicitamente come coverage
    0.0, mai lasciato disallineare silenziosamente il conteggio mascherato da quello totale.
    """
    # Immagine "contatore": vale 1 in ogni punto del cervello. int32, non uint8: con una
    # casella troppo stretta il conteggio va in overflow silenzioso (bug verificato su dati reali,
    # vedi docs/debugging/debug_23_07_26.md).
    ones_img = nib.Nifti1Image(np.ones(atlas_img.shape, dtype=np.int32), atlas_img.affine, atlas_img.header)

    masker_total = NiftiLabelsMasker(labels_img=atlas_img, background_label=0, strategy="sum", standardize=False)
    n_total = np.squeeze(masker_total.fit_transform(ones_img))
    total_by_label = dict(zip(masker_total.labels_[1:], n_total))  # [1:]: labels_[0] e' il placeholder "Background"

    masker_healthy = NiftiLabelsMasker(
        labels_img=atlas_img, mask_img=healthy_img, background_label=0, strategy="sum", standardize=False
    )
    n_healthy = np.squeeze(masker_healthy.fit_transform(ones_img))
    healthy_by_label = dict(zip(masker_healthy.labels_[1:], np.atleast_1d(n_healthy)))

    # .get(lbl, 0.0): se una zona e' sparita dal secondo conteggio (100% lesionata), il suo valore e' 0.0 - mai un errore.
    return np.array([healthy_by_label.get(lbl, 0.0) / total_by_label[lbl] for lbl in label_ids])


def vectorize_upper_triangle(matrix_df, node_names):
    """Solo le connessioni uniche (triangolo superiore, diagonale esclusa), come vettore etichettato.

    La matrice FC e' simmetrica (A-B == B-A) e ha 1.0 sulla diagonale (un nodo correlato con se stesso) -
    tenerla intera duplicherebbe ogni valore e aggiungerebbe una diagonale non informativa.
    """
    n = len(node_names)
    row_idx, col_idx = np.triu_indices(n, k=1)  # k=1: esclude la diagonale
    edge_names = [f"{node_names[i]}__{node_names[j]}" for i, j in zip(row_idx, col_idx)]
    values = matrix_df.values[row_idx, col_idx]
    return pd.Series(values, index=edge_names)


def process_subject(subject):
    """Fase A completa per un paziente: lesione -> coverage -> masking (NaN) -> salvataggio -> vettorizzazione.

    Ritorna (edge_vector, n_compromised): il vettore vettorizzato e quante zone sono state marcate NaN,
    per il report di qualita' della Fase B.
    """
    lesion_path = (
        f"{DATA_ROOT}/manual_masks/{subject}/anat/"
        f"{subject}_space-MNI152NLin6Asym_label-lesion_mask.nii.gz"
    )
    lesion_img = nib.load(lesion_path)
    # stessa shape non implica stesso orientamento spaziale (vedi affine invertito, sezione successiva) -
    # resample esplicito via affine, mai un allineamento assunto.
    lesion_resampled = resample_to_img(
        lesion_img, atlas_img, interpolation="nearest", force_resample=True, copy_header=True
    )
    lesion_data = (np.asarray(lesion_resampled.get_fdata()) > 0.5).astype(np.int32)

    healthy_img = nib.Nifti1Image((1 - lesion_data), atlas_img.affine, atlas_img.header)
    parcel_coverage = compute_parcel_coverage(atlas_img, label_ids, healthy_img)
    compromised = parcel_coverage < MIN_COVERAGE
    compromised_names = node_names[compromised]

    fc_path = glob.glob(f"{DATA_ROOT}/features/{subject}/func/*{ATLAS_COMBO}*.csv")[0]
    fc = pd.read_csv(fc_path, sep="\t", index_col=0)
    assert list(fc.index) == list(node_names), f"{subject}: ordine nodi FC non combacia con l'atlante"

    fc_masked = fc.copy()  # non modifichiamo mai i dati originali
    fc_masked.loc[compromised_names, :] = np.nan  # NaN, non zero: il valore concreto si decide solo a valle
    fc_masked.loc[:, compromised_names] = np.nan

    # Salvataggio su disco della matrice mascherata COSI' COM'E' - prima di vettorizzare, formato ispezionabile.
    masked_path = MASKED_FC_ROOT / f"{subject}_masked_fc.csv"
    fc_masked.to_csv(masked_path)

    edge_vector = vectorize_upper_triangle(fc_masked, node_names)
    return edge_vector, len(compromised_names), masked_path


results = {}
for subject in SUBJECTS:
    edge_vector, n_compromised, masked_path = process_subject(subject)
    results[subject] = edge_vector
    print(f"{subject}: {n_compromised} zone compromesse su {len(node_names)} -> salvato in {masked_path}")

sub-STUNIPD0002: 0 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0002_masked_fc.csv


sub-STUNIPD0003: 8 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0003_masked_fc.csv


sub-STUNIPD0006: 9 zone compromesse su 239 -> salvato in ../data/derived/features/masked_fc/atlas-Yan200TianS2Buckner7N/sub-STUNIPD0006_masked_fc.csv


## Fase B — tutti i pazienti insieme

Continuando l'esempio giocattolo: ogni paziente ha zone diverse compromesse (dipende da dove ha la lesione), ma le **etichette delle 6 connessioni devono restare le stesse** per tutti, nello stesso ordine — solo cosi' si possono mettere un paziente per riga nella stessa tabella.

1. **Verifica l'allineamento**: tutte le etichette (`A__B`, `A__C`, ...) devono combaciare esattamente tra pazienti.
2. **Impila**: un paziente per riga -> tabella pazienti x connessioni, con NaN sparsi in posizioni diverse per ciascuno.
3. **Report di qualita'**: quante connessioni sono NaN per ciascun paziente, e quante volte ciascuna connessione e' NaN nel campione.
4. **Salva su disco** questa matrice impilata (ancora con i NaN dentro) in `data/derived/features/fc_matrix/` — l'imputazione (NaN -> 0 o altro) resta un passo separato, fatto solo quando serve davvero per la riduzione dimensionale, non qui.

In [4]:
# 1. Verifica allineamento: tutti i pazienti devono avere le stesse identiche etichette di connessione.
reference_edges = results[SUBJECTS[0]].index
for subject, edge_vector in results.items():
    assert list(edge_vector.index) == list(reference_edges), f"{subject}: etichette di connessione disallineate"

# 2. Impila: un paziente per riga.
fc_matrix = pd.DataFrame({subject: vec for subject, vec in results.items()}).T
fc_matrix.index.name = "subject_id"
print(f"Matrice impilata: {fc_matrix.shape[0]} pazienti x {fc_matrix.shape[1]} connessioni\n")

# 3. Report di qualita' - mai calcolato e poi scartato.
nan_per_subject = fc_matrix.isna().sum(axis=1)
nan_per_edge = fc_matrix.isna().sum(axis=0)
print("NaN per paziente:")
print(nan_per_subject)
print(f"\nConnessioni mai compromesse in nessun paziente di questo campione: {(nan_per_edge == 0).sum()} / {len(nan_per_edge)}")
print(f"Connessione con piu' NaN nel campione: '{nan_per_edge.idxmax()}' ({nan_per_edge.max()} pazienti su {len(SUBJECTS)})")

# 4. Salvataggio della matrice impilata, ancora con i NaN - destinazione concordata.
fc_matrix_path = FC_MATRIX_ROOT / "fc_matrix_demo.csv"
fc_matrix.to_csv(fc_matrix_path)
print(f"\nMatrice impilata salvata in: {fc_matrix_path}")

Matrice impilata: 3 pazienti x 28441 connessioni

NaN per paziente:
subject_id
sub-STUNIPD0002       0
sub-STUNIPD0003    1876
sub-STUNIPD0006    2335
dtype: int64

Connessioni mai compromesse in nessun paziente di questo campione: 24310 / 28441
Connessione con piu' NaN nel campione: '7Networks_LH_SalVentAttn_Ins_1__7Networks_RH_Cont_PFCl_4' (2 pazienti su 3)

Matrice impilata salvata in: ../data/derived/features/fc_matrix/atlas-Yan200TianS2Buckner7N/fc_matrix_demo.csv


## Fase C — imputazione (solo un'anteprima, non fatta qui)

Questo notebook si ferma alla matrice con i NaN — l'imputazione (NaN -> 0 o un'altra strategia) e' un passo **separato**, deliberatamente non incluso in questo output, perche' va fatto solo subito prima di un metodo che non tollera NaN (PCA/UMAP), non prima. La cella sotto e' solo un'anteprima di cosa succederebbe con la strategia piu' semplice (`"zero"`, la stessa usata da Siegel/Griffis), per mostrare l'effetto senza salvarlo.

In [5]:
n_nan_before = int(fc_matrix.isna().sum().sum())
fc_matrix_imputed_preview = fc_matrix.fillna(0.0)  # anteprima soltanto - non salvata
print(f"Valori NaN nella matrice impilata: {n_nan_before} (su {fc_matrix.size} totali)")
print(f"Dopo imputazione a zero (anteprima, non salvata): {int(fc_matrix_imputed_preview.isna().sum().sum())} NaN rimasti")

Valori NaN nella matrice impilata: 4211 (su 85323 totali)
Dopo imputazione a zero (anteprima, non salvata): 0 NaN rimasti


## Prossimo passo

Questo e' un **prototipo**, validato per ora su 3 pazienti e una sola versione della mappa cerebrale (ce ne sono 12 in totale: `Yan{100,200,300,400}TianS{1,2,3}Buckner7N`). Il percorso completo concordato, da qui a una vera pipeline in `src/`:

1. **Fase A + B** (questo notebook) -> `src/features/functional.py`, tipizzato e testato, con un test di regressione per ciascuno dei due bug gia' trovati (overflow `uint8`, zona che sparisce).
2. **Fase C** (imputazione) -> funzione separata e configurabile (`NAN_IMPUTATION_STRATEGIES`, default `"zero"`), richiamata solo da `dim_reduction.py`, mai dentro la costruzione della feature - con log esplicito di quanti valori sono stati imputati.
3. **Fase D** (controllo di robustezza, facoltativo ma consigliato) - ripetere la riduzione dimensionale/clustering solo sui pazienti senza nessuna connessione compromessa, confrontare con il risultato sull'intero campione (stesso controllo fatto da Griffis et al. 2019).
4. Nuovo entry point `src/pipeline/build_fc_matrix.py` + `config/pipelines/build_fc_matrix.json` + `jobs/run_build_fc_matrix.sh`, con gli output nelle due destinazioni gia' usate qui: `data/derived/features/masked_fc/` e `data/derived/features/fc_matrix/`.